In [ ]:
import pandas

In [ ]:
df = pandas.read_csv('data.csv')

In [ ]:
df.head(5)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [ ]:
df.shape

(10000, 14)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [ ]:
df.isnull().sum()

,0
UDI,0
Product ID,0
Type,0
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],0
Torque [Nm],0
Tool wear [min],0
Machine failure,0
TWF,0


In [ ]:
df['Machine failure'].value_counts()/df.shape[0]*100

,count
Machine failure,
0,96.61
1,3.39


# Model Training Part 1 --> no balanced data

Train Test Split

In [ ]:
import sklearn as sk

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [ ]:
X = df[['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']]
y = df['Machine failure']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42, stratify=y)

In [ ]:
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("Training failure ratio:")
print(y_train.value_counts(normalize=True) * 100)

print("Testing failure ratio:")
print(y_test.value_counts(normalize=True) * 100)

Training shape: (8000, 5)
Testing shape: (2000, 5)
Training failure ratio:
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64
Testing failure ratio:
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


Scaling --> after train-test split

bcoz of data leakage

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
pipeline = Pipeline([

    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])


In [ ]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)


In [ ]:
print(accuracy * 100)
print(precision * 100)
print(recall * 100)

96.85000000000001
69.23076923076923
13.23529411764706


Threshold tuning

bcoz we need to balance Recall & Precision

In [ ]:
probabilities = pipeline.predict_proba(X_test)

print(probabilities[:,1])
print(pipeline.classes_)

[0.56343547 0.02610996 0.0442577  ... 0.00747081 0.05744253 0.00120861]
[0 1]


In [ ]:
import numpy as np
values = np.array(probabilities[:,1])
threshold = 0.3
var = values >= threshold #produces true or false
y_pred_03 = var.astype(int)
print(y_pred)


[1 0 0 ... 0 0 0]


In [ ]:
# after
precision_score(y_test, y_pred_03)
recall_score(y_test, y_pred_03)
confusion_matrix(y_test, y_pred_03)

array([[1913,   19],
       [  46,   22]])

In [ ]:
# after
print(recall_score(y_test, y_pred_03) * 100)
print(precision_score(y_test, y_pred_03) * 100)

32.35294117647059
53.65853658536586


In [ ]:
# before
print(np.sum(y_pred != y_pred_03))

0


In [ ]:
# before
print("Original:", np.sum(y_pred == 1))
print("Threshold 0.3:", np.sum(y_pred_03 == 1))

Original: 13
Threshold 0.3: 13


In [ ]:
# after
print("Original:", np.sum(y_pred == 1))
print("Threshold 0.3:", np.sum(y_pred_03 == 1))

print("Changed predictions:", np.sum(y_pred != y_pred_03))

Original: 13
Threshold 0.3: 41
Changed predictions: 28


In [ ]:
# before
print("Min probability:", values.min())
print("Max probability:", values.max())

print("Between 0.3 and 0.5:",
      np.sum((values >= 0.3) & (values < 0.5)))

Min probability: 6.472679187619612e-05
Max probability: 0.938987990609455
Between 0.3 and 0.5: 28


Train --> validation --> test

what we did above was validation not actual test

We want:
- 70% training
- 15% validation
- 15% testing

In [ ]:
# training = 70% and testing = 30%
X_train,X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# training = 70% , validation = 15% and testing = 15%
X_val, X_test, y_val, y_test  = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [ ]:
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (7000, 5) (7000,)
Validation: (1500, 5) (1500,)
Test: (1500, 5) (1500,)


Threshold comparison on Validation data

not on test data

test data remains untouched

In [ ]:
probabilities = pipeline.predict_proba(X_val)

print(probabilities[:,1])
print(pipeline.classes_)

NotFittedError: Pipeline is not fitted yet.

In [ ]:
thresholds = [0.2, 0.3, 0.4]
values = probabilities[:,1]

for threshold in thresholds:
    # Convert probabilities to 0 and 1
    values_01 = (values >= threshold).astype(int)

    # Calculate precision
    precision_score(y_test, values_01)

    # Calculate recall
    recall_score(y_test, values_01)

    # Print results
    print("Threshold:", threshold)
    print("Precision:", precision_score(y_test, values_01))
    print("Recall:", recall_score(y_test, values_01))
    print()

Threshold: 0.2
Precision: 0.4230769230769231
Recall: 0.43137254901960786

Threshold: 0.3
Precision: 0.5483870967741935
Recall: 0.3333333333333333

Threshold: 0.4
Precision: 0.7647058823529411
Recall: 0.2549019607843137



all 3 thresholds are not good

To balance precision & recall we will use F1-score


In [ ]:
from sklearn.metrics import f1_score

In [ ]:
thresholds = [0.2, 0.3, 0.4]
values = probabilities[:,1]

for threshold in thresholds:
    # Convert probabilities to 0 and 1
    values_01 = (values >= threshold).astype(int)

    # Calculate precision
    precision_score(y_test, values_01)

    # Calculate recall
    recall_score(y_test, values_01)

    # calculate f1 score
    f1_score(y_test, values_01)

    # Print results
    print("Threshold:", threshold)
    print("Precision:", precision_score(y_test, values_01))
    print("Recall:", recall_score(y_test, values_01))
    print("F1-score:", f1_score(y_test, values_01))
    print()

Threshold: 0.2
Precision: 0.4230769230769231
Recall: 0.43137254901960786
F1-score: 0.42718446601941745

Threshold: 0.3
Precision: 0.5483870967741935
Recall: 0.3333333333333333
F1-score: 0.4146341463414634

Threshold: 0.4
Precision: 0.7647058823529411
Recall: 0.2549019607843137
F1-score: 0.38235294117647056



threshol = 0.2 wins

but at what cost?

precision is 42% means our model will generate more false alarms

**Therefore we will now balance both classes(normal & failure)**

# Model Training Part 2 --> with balanced data

at threshold = 0.5

In [ ]:
# run train-val-test
# training = 70% and testing = 30%
X_train,X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# training = 70% , validation = 15% and testing = 15%
X_val, X_test, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [ ]:
pipeline = Pipeline([

    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])


In [ ]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [ ]:
probabilities = pipeline.predict_proba(X_val)

print(probabilities[:,1])
print(pipeline.classes_)

[0.72343165 0.41399869 0.65014288 ... 0.54344852 0.82447089 0.00565142]
[0 1]


In [ ]:
import numpy as np
values = np.array(probabilities[:,1])
threshold = 0.5
var = values >= threshold #produces true or false
y_pred_05 = var.astype(int)
print(y_pred_05)


[1 0 1 ... 1 1 0]


In [ ]:
precision_score(y_test, y_pred_05) *100

13.602941176470587

In [ ]:
recall_score(y_test, y_pred_05)*100


72.54901960784314

Model	Precision	Recall

Standard Logistic Regression, threshold 0.2	42.31%	43.14%
Balanced Logistic Regression, threshold 0.5	13.60%	72.55%

The balanced model detects more failures but generates substantially more false alarms. These results give us useful baselines, although **we still need a verified, like-for-like comparison using the same validation set.**

Therefore, we will test **Random Forest classifier**

# Model Training --> Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# training = 70% and testing = 30%
X_train,X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# training = 70% , validation = 15% and testing = 15%
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [ ]:
pipeline = Pipeline(
    [
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(random_state=42, n_estimators=100))
    ]
)

In [ ]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', RandomForestClassifier(random_state=42))])

In [ ]:
probabilities = pipeline.predict_proba(X_val)

print(probabilities[:,1])
print(pipeline.classes_)

[0. 0. 0. ... 0. 0. 0.]
[0 1]


In [ ]:
import numpy as np
values = np.array(probabilities[:,1])
threshold = 0.5
var = values >= threshold #produces true or false
y_pred_05_random = var.astype(int)
print(y_pred_05_random)


[0 0 0 ... 0 0 0]


In [ ]:
precision_score(y_test, y_pred_05_random) *100

87.8048780487805

In [ ]:
recall_score(y_test, y_pred_05_random)*100


70.58823529411765

In [ ]:
train_pred = pipeline.predict(X_train)
val_pred = pipeline.predict(X_val)

In [ ]:
accuracy_score(y_train, train_pred)

1.0

In [ ]:
accuracy_score(y_val, val_pred)

0.94

In [ ]:
precision_score(y_train, train_pred)

1.0

In [ ]:
precision_score(y_val, val_pred)

0.024390243902439025

In [ ]:
recall_score(y_train, train_pred)

1.0

In [ ]:
recall_score(y_val, val_pred)

0.0196078431372549

In [ ]:
f1_score(y_train, train_pred)

1.0

In [ ]:
f1_score(y_val, val_pred)

0.021739130434782608

In [ ]:
from sklearn.metrics import confusion_matrix

print("Training:")
print(confusion_matrix(y_train, train_pred))

print("Validation:")
print(confusion_matrix(y_val, val_pred))

Training:
[[6763    0]
 [   0  237]]
Validation:
[[1409   40]
 [  50    1]]


In [ ]:
print("Actual failures:", sum(y_val == 1))
print("Predicted failures:", sum(val_pred == 1))

Actual failures: 51
Predicted failures: 41


In [ ]:
print(type(pipeline.named_steps["model"]))

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

print("Training labels:", len(y_train))
print("Validation labels:", len(y_val))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
Training rows: 7000
Validation rows: 1500
Training labels: 7000
Validation labels: 1500
